# Part 2 — Accelerated relaxation solver and elliptical force map

Replaces the exact matrix inversion with iterative relaxation (neighbor averaging) compiled with Numba `njit`, and draws the surface / force map for rectangular or elliptical boundaries (Figure 3b in the manuscript).

> **Note.** This notebook imports the `surface` module (the relaxation solver). Keep `surface.py` (provided in this folder) in the same directory, or run from `part2/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from surface import surface, make_rect_edge, make_ellipse_edge

def force_map(nx, ny, dot_pos, dot_height, ellipse=False, show_ellipse_edge=False):
    """
    Compute the force map based on surface height differences.
    If ellipse=True, use elliptical boundary; otherwise, rectangular.
    """
    # Choose boundary type
    edges = make_ellipse_edge(nx, ny) if ellipse else make_rect_edge(nx, ny)

    # Get height map from surface solver
    height = surface(nx, ny, dot_pos, dot_height, edges)

    # Compute force = gradient magnitude
    gy, gx = np.gradient(height)
    force = np.sqrt(gx**2 + gy**2)

    # ---------------------------
    # Plotting
    # ---------------------------
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))

    # Surface plot
    surf = ax.plot_surface(
        X, Y, force,
        cmap=cm.viridis,
        linewidth=0,
        antialiased=True,
        alpha=0.9
    )

    # Add ellipse boundary if enabled
    if ellipse and show_ellipse_edge:
        cx, cy = (nx - 1) / 2, (ny - 1) / 2
        rx, ry = cx, cy
        theta = np.linspace(0, 2 * np.pi, 400)
        ex = cx + rx * np.cos(theta)
        ey = cy + ry * np.sin(theta)

        # Nearest-neighbor sampling of force values along ellipse
        ix = np.clip(np.rint(ex).astype(int), 0, nx - 1)
        iy = np.clip(np.rint(ey).astype(int), 0, ny - 1)
        ez = force[iy, ix]

        ax.plot(ex, ey, ez, color="red", linewidth=2, label="Ellipse Edge")
        ax.legend()

    ax.set_title("Force Map" + (" (ellipse)" if ellipse else ""))
    ax.set_xlabel("X axis")
    ax.set_ylabel("Y axis")
    ax.set_zlabel("Force")

    plt.tight_layout()
    plt.show()

    return force

if __name__ == "__main__":
    force_map(30, 30, dot_pos=(15, 15), dot_height=1.0, ellipse=True, show_ellipse_edge=True)
